# Експеримент 2: contrastive fine-tuning (MiniLM)

**Ідея.** Готова модель не знає, які навички належать до яких посад у ESCO. Ми донавчаємо її на позитивних парах
«посада → навичка» з `train.csv`, щоб вектори пов'язаних посад і навичок зблизились.

**Метод.** `MultipleNegativesRankingLoss`: у кожному батчі для посади «правильна» навичка — її пара, а навички з інших пар батчу
слугують негативами. `NO_DUPLICATES` sampler гарантує, що одна й та сама посада чи навичка не потрапить у батч двічі.

**Дані.** Беремо лише пари з `label > 0` (essential та optional). Невипадкові негативи тут не потрібні — вони з'являються з батчу.
Валідацію (5% посад) відрізаємо **за назвою посади**, щоб модель не бачила їх під час навчання; dev-набір для підбору налаштувань **не використовуємо**.

**Оцінювання.** Після навчання — той самий протокол, що й для baseline: dev, ранжування всього корпусу, `ranking_metrics.py`.

> **Спочатку запусти з `SMOKE = True`** (4 000 пар, 1 епоха) — це перевірка, що все працює, і замір швидкості на твоєму Mac.
> Потім постав `SMOKE = False` і запусти повноцінне навчання.

In [8]:
import os, sys
from pathlib import Path

# ноутбук лежить в experiments/, а config.py та ranking_metrics.py — в корені проєкту
if not Path("config.py").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Робоча папка:", os.getcwd())

Робоча папка: /Users/mac/spj/project/Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches


In [9]:
import ast
import json

import numpy as np
import pandas as pd
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import BatchSamplers

from config import RANDOM_SEED
from ranking_metrics import evaluate_query, save_results

TRAIN_PATH = Path("data/processed/train.csv")
VAL_DIR = Path("data/raw/TaskB/validation")

# ---------- налаштування ----------
BASE_MODEL = "all-MiniLM-L6-v2"
OUTPUT_DIR = Path("models/minilm_mnrl")   # ваги моделі (папка models/ має бути в .gitignore!)
RESULT_NAME = "finetuned_minilm_mnrl"
EPOCHS = 2
BATCH_SIZE = 128
LEARNING_RATE = 2e-5
VAL_FRACTION = 0.05
SMOKE = False   # True = швидка перевірка (4000 пар, 1 епоха, результат не зберігається)

## 1. Підготовка даних
Позитивні пари, розбиття на train/val **за посадою**.

In [10]:
train_all = pd.read_csv(TRAIN_PATH)
pos = train_all[train_all["label"] > 0][["job_name", "skill_name"]].drop_duplicates().reset_index(drop=True)
print(f"позитивних пар: {len(pos)}, посад: {pos['job_name'].nunique()}, навичок: {pos['skill_name'].nunique()}")

rng = np.random.RandomState(RANDOM_SEED)
jobs = np.array(sorted(pos["job_name"].unique()))
n_val = max(1, int(len(jobs) * VAL_FRACTION))
val_jobs = set(rng.choice(jobs, size=n_val, replace=False))

val_pairs = pos[pos["job_name"].isin(val_jobs)].reset_index(drop=True)
train_pairs = pos[~pos["job_name"].isin(val_jobs)].reset_index(drop=True)

if SMOKE:
    train_pairs = train_pairs.sample(n=min(4000, len(train_pairs)), random_state=RANDOM_SEED).reset_index(drop=True)
print(f"train: {len(train_pairs)} пар | val: {len(val_pairs)} пар ({len(val_jobs)} посад)")

train_dataset = Dataset.from_dict({
    "anchor": train_pairs["job_name"].astype(str).tolist(),
    "positive": train_pairs["skill_name"].astype(str).tolist(),
})

позитивних пар: 114686, посад: 3011, навичок: 13224
train: 108870 пар | val: 5816 пар (150 посад)


## 2. Валідація під час навчання
`InformationRetrievalEvaluator`: для валідаційних посад ранжуємо **усі** навички з train і дивимось nDCG@10, MAP тощо. Це лише для контролю, не для офіційної оцінки.

In [11]:
all_skills = sorted(pos["skill_name"].astype(str).unique())
skill_id = {s: f"s{i}" for i, s in enumerate(all_skills)}
val_job_list = sorted(val_jobs)
job_id = {j: f"j{i}" for i, j in enumerate(val_job_list)}

relevant = {}
for r in val_pairs.itertuples(index=False):
    relevant.setdefault(job_id[r.job_name], set()).add(skill_id[str(r.skill_name)])

val_evaluator = InformationRetrievalEvaluator(
    queries={job_id[j]: str(j) for j in val_job_list},
    corpus={skill_id[s]: s for s in all_skills},
    relevant_docs=relevant,
    name="val",
    ndcg_at_k=[10, 100],
    map_at_k=[100],
    mrr_at_k=[10],
    show_progress_bar=False,
)

## 3. Навчання
На Mac з Apple Silicon бібліотека сама використає GPU (`mps`). Нижче надрукується пристрій.

In [12]:
model = SentenceTransformer(BASE_MODEL)
print("device:", model.device)

epochs = 1 if SMOKE else EPOCHS
loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="models/checkpoints",
    num_train_epochs=epochs,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    seed=RANDOM_SEED,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=val_evaluator,
)
trainer.train()

if not SMOKE:
    OUTPUT_DIR.parent.mkdir(exist_ok=True)
    model.save_pretrained(str(OUTPUT_DIR))
    print("модель збережено:", OUTPUT_DIR)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


device: mps:0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/Users/mac/spj/project/Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Val Cosine Accuracy@1,Val Cosine Accuracy@3,Val Cosine Accuracy@5,Val Cosine Accuracy@10,Val Cosine Precision@1,Val Cosine Precision@3,Val Cosine Precision@5,Val Cosine Precision@10,Val Cosine Recall@1,Val Cosine Recall@3,Val Cosine Recall@5,Val Cosine Recall@10,Val Cosine Ndcg@10,Val Cosine Ndcg@100,Val Cosine Mrr@10,Val Cosine Map@100
1,2.927031,No log,0.593333,0.793333,0.893333,0.926667,0.593333,0.475556,0.421333,0.343333,0.017382,0.041144,0.060986,0.098894,0.390508,0.356523,0.705730,0.145664
2,2.760810,No log,0.573333,0.813333,0.893333,0.933333,0.573333,0.488889,0.433333,0.352000,0.016684,0.042727,0.061995,0.101877,0.398397,0.371556,0.703175,0.154843


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

модель збережено: models/minilm_mnrl


## 4. Оцінювання на dev (офіційний протокол)
Ранжуємо весь корпус із 9 052 навичок для кожної посади, навичка представлена першим аліасом (як у навчанні).

In [13]:
queries = pd.read_csv(VAL_DIR / "queries", sep="\t").rename(columns={"jobtitle": "job_name", "q_id": "job_id"})
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t").rename(columns={"c_id": "skill_id"})
corpus["skill_aliases"] = corpus["skill_aliases"].apply(ast.literal_eval)
corpus["skill_name"] = corpus["skill_aliases"].apply(lambda x: x[0])
qrels = pd.read_csv(VAL_DIR / "qrels.tsv", header=None, sep="\t", names=["job_id", "_", "skill_id", "label"])

skill_index = {sid: i for i, sid in enumerate(corpus["skill_id"])}
grades_by_job = {}
for r in qrels.itertuples(index=False):
    grades_by_job.setdefault(r.job_id, np.zeros(len(corpus)))[skill_index[r.skill_id]] = r.label


def encode_texts(texts):
    unique = sorted(set(texts))
    vecs = model.encode(unique, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    return dict(zip(unique, vecs))


job_emb = encode_texts(queries["job_name"].tolist())
skill_map = encode_texts(corpus["skill_name"].tolist())
skill_emb = np.vstack([skill_map[s] for s in corpus["skill_name"]])

per_query = []
for row in queries.itertuples(index=False):
    if row.job_id in grades_by_job:
        scores = skill_emb @ job_emb[row.job_name]
        per_query.append(evaluate_query(scores, grades_by_job[row.job_id]))

mean = {k: round(float(np.mean([q[k] for q in per_query])), 4) for k in per_query[0]}
print("dev:", mean)

if SMOKE:
    print("SMOKE-режим: результат НЕ збережено. Постав SMOKE = False для повного навчання.")
else:
    save_results(
        RESULT_NAME,
        f"Fine-tuned {BASE_MODEL} (MultipleNegativesRankingLoss, NO_DUPLICATES, {epochs} епох, batch {BATCH_SIZE}, "
        f"lr {LEARNING_RATE}) на позитивних парах train.csv (essential+optional); валідація: {VAL_FRACTION:.0%} посад за назвою. "
        f"Cosine similarity, навичка = перший аліас.",
        per_query,
    )

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/140 [00:00<?, ?it/s]

dev: {'ndcg_graded': 0.6744, 'ndcg_binary': 0.7217, 'map_binary': 0.2273, 'mrr_binary': 0.6643, 'p@1': 0.5, 'p@5': 0.4704, 'p@10': 0.4441}
Збережено: results/finetuned_minilm_mnrl.json


## 5. Порівняння з усіма підходами

In [14]:
rows = []
for p in sorted(Path("results").glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"approach": p.stem, **{k: v for k, v in d.items() if k != "description"}})

(pd.DataFrame(rows).sort_values("ndcg_graded", ascending=False).reset_index(drop=True)
 if rows else "results/ порожня")

,approach,ndcg_graded,ndcg_binary,map_binary,mrr_binary,p@1,p@5,p@10
0,cosine_jobbert_mean,0.7381,0.7889,0.3416,0.7886,0.6480,0.5632,0.5178
1,cosine_jobbert_first,0.7359,0.7899,0.3479,0.8210,0.6974,0.5645,0.4914
2,cosine_jobbert_max,0.7223,0.7723,0.3077,0.7535,0.5987,0.5125,0.4816
3,finetuned_minilm_mnrl,0.6744,0.7217,0.2273,0.6643,0.5000,0.4704,0.4441
4,cosine_mpnet_mean,0.6327,0.6763,0.1612,0.6302,0.4539,0.3980,0.3711
5,cosine_mpnet_first,0.6251,0.6704,0.1545,0.6135,0.4408,0.3809,0.3549
6,cosine_minilm_mean,0.6214,0.6645,0.1477,0.6397,0.4803,0.3836,0.3658
7,cosine_mpnet_max,0.6189,0.6625,0.1446,0.6059,0.4178,0.3783,0.3523
8,cosine_minilm,0.6154,0.6599,0.1430,0.6206,0.4342,0.3829,0.3539
9,cosine_minilm_first,0.6154,0.6599,0.1430,0.6206,0.4342,0.3829,0.3539
